In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics -q
!pip install torch torchvision -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

MODEL_MULTI = '/content/drive/MyDrive/YOLO without P5 head/runs/yolov8m_multiclass_clspw_0.5_multiscale_0.5/weights/best.pt'

model_multi = YOLO(MODEL_MULTI)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
%cd "/content/drive/MyDrive/YOLO without P5 head"

/content/drive/MyDrive/YOLO without P5 head


In [ ]:
import shutil, os

DRIVE_DATASET = '/content/drive/MyDrive/YOLO without P5 head/Dataset_Curat_YOLO'
DRIVE_YAMLS   = '/content/drive/MyDrive/YOLO without P5 head'

COLAB_DATASET = '/content/dataset'

if os.path.exists(COLAB_DATASET):
    shutil.rmtree(COLAB_DATASET)
shutil.copytree(DRIVE_DATASET, COLAB_DATASET)

for f in ['data.yaml', 'imcmd_yolov8m_full.yaml', 'imcmd_modules.py']:
    src = os.path.join(DRIVE_YAMLS, f)
    if os.path.exists(src):
        shutil.copy(src, f'/content/{f}')
    else:
        print(f"[WARN] Not found: {src}")

for split in ['train', 'val', 'test']:
    imgs = len(os.listdir(f'{COLAB_DATASET}/{split}/images'))
    lbls = len(os.listdir(f'{COLAB_DATASET}/{split}/labels'))
    print(f"  {split:>5}: {imgs} images, {lbls} labels")

  train: 1672 images, 1672 labels
    val: 358 images, 358 labels
   test: 358 images, 358 labels


In [ ]:
yaml_content = """path: /content/dataset
train: train/images
val:   val/images
test:  test/images

nc: 4
names: ['comedo 0', 'nodul 3', 'papule 1', 'pustule 2']
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)

import os
for split in ['train', 'val', 'test']:
    path = f'/content/dataset/{split}/images'
    if os.path.exists(path):
        print(f"  {split}: {len(os.listdir(path))} images")
    else:
        print(f"  [WARN] Missing: {path}")

  train: 1672 images
  val: 358 images
  test: 358 images


# **Grid Search YOLO Multiclass multiscale + cls_pw**

In [ ]:
import pandas as pd

CONFS = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
IOUS  = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60,]

results = []
print(f"Grid search YOLO multiclass: {len(CONFS)*len(IOUS)} combinatii pe VALIDARE\n")

for conf in CONFS:
    for iou in IOUS:
        m = model_multi.val(
            data='/content/data.yaml',
            split='val',
            imgsz=640,
            conf=conf,
            iou=iou,
            verbose=False,
            plots=False,
        )
        P  = m.box.mp      # medie precizie
        R  = m.box.mr      # medie recall

        f1 = 0 # optimizez modelul dupa media armonica
        if (P+R) > 0:
          f1 = 2*P*R/(P+R)

        results.append({
            'conf': conf, 'iou': iou,
            'P': P, 'R': R, 'F1': f1,
            'mAP50': m.box.map50, 'mAP50_95': m.box.map,
        })
        print(f"  conf={conf:.2f} iou={iou:.2f} | F1={f1:.4f} P={P:.4f} R={R:.4f} mAP50={m.box.map50:.4f}")

df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print("\nTOP 5 dupa F1")
print(df.head(5).to_string(index=False))

best = df.iloc[0]
BEST_CONF = float(best['conf'])
BEST_IOU  = float(best['iou'])
print(f"\n Cel mai bun model: conf={BEST_CONF}, iou={BEST_IOU}, F1={best['F1']:.4f}")

Grid search YOLO multiclass: 81 combinatii pe VALIDARE

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 740.8±300.4 MB/s, size: 23.3 KB)
val: Scanning /content/dataset/val/labels... 358 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 358/358 1.4Kit/s 0.3s
val: New cache created: /content/dataset/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.5s
                   all        358       4495      0.354      0.328      0.245     0.0828
Speed: 1.0ms preprocess, 18.3ms inference, 0.0ms loss, 0.9ms postprocess per image
  conf=0.01 iou=0.10 | F1=0.3406 P=0.3541 R=0.3280 mAP50=0.2450
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1468.4±658.5 MB/s, 

# **Evaluare YOLO Multiclass: cls_pw + multiscale cu conf 0.01, iou = 0.3 pe TEST set**

In [ ]:
print("EVALUARE FINALA PE TEST\n")

for tta in [False, True]:
    m = model_multi.val(
        data='/content/data.yaml',
        split='test',
        imgsz=640,
        conf=BEST_CONF,
        iou=BEST_IOU,
        augment=tta,
        verbose=False,
        plots=False,
    )
    P  = m.box.mp
    R  = m.box.mr
    tag = "CU TTA " if tta else "FARA TTA"
    print(f"[{tag}] P={P:.4f}  R={R:.4f}  mAP50={m.box.map50:.4f}  mAP50-95={m.box.map:.4f}")
    print(f"          Per clasa mAP50: {[round(x,3) for x in m.box.maps]}\n")

EVALUARE FINALA PE TEST

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 919.3±453.6 MB/s, size: 35.4 KB)
val: Scanning /content/dataset/test/labels... 358 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 358/358 673.5it/s 0.5s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.2it/s 10.3s
                   all        358       3422      0.385      0.388      0.304      0.107
Speed: 1.0ms preprocess, 23.9ms inference, 0.0ms loss, 1.0ms postprocess per image
[FARA TTA] P=0.3850  R=0.3877  mAP50=0.3042  mAP50-95=0.1075
          Per clasa mAP50: [np.float64(0.118), np.float64(0.089), np.float64(0.111), np.float64(0.111)]

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1076.5±294.0 MB/s, size: 2

In [ ]:
import argparse, json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict

import cv2
import torch
import torch.nn.functional as F
from torchvision import transforms
import timm

CLASS_NAMES    = ["comedo", "nodul", "papule", "pustule"]
NUM_CLASSES    = 4
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

CONTEXT   = 0.20
CROP_SIZE = 224

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

IOU_THRESH_50     = 0.50
IOU_THRESHOLDS_95 = np.arange(0.50, 1.00, 0.05)


In [ ]:
def make_square_crop(img_np, x1, y1, x2, y2, context=CONTEXT, size=CROP_SIZE):
    # Calculez latimea si inaltimea imaginii:
    img_h, img_w = img_np.shape[:2]
    # calculez latimea si inaltimea ferestrei:
    bw = x2 - x1; bh = y2 - y1
    # adaug mai mult context, acel 20% pentru a nu fii foarte mici leziunile:
    pad_x = bw * context; pad_y = bh * context
    x1e = x1 - pad_x; y1e = y1 - pad_y
    x2e = x2 + pad_x; y2e = y2 + pad_y

    # il transform in patrat, calculez latura cea mai mare:
    cw = x2e - x1e; ch = y2e - y1e
    side = max(cw, ch)

    # centrez patratul:
    cx = (x1e + x2e) / 2; cy = (y1e + y2e) / 2
    x1s = cx - side / 2; y1s = cy - side / 2
    x2s = cx + side / 2; y2s = cy + side / 2

    # verific daca patratul iese din imagine, daca iese, adaug pixeli negrii in zona unde iese:
    pl = max(0.0, -x1s); pt = max(0.0, -y1s)
    pr = max(0.0, x2s - img_w); pb = max(0.0, y2s - img_h)
    x1c = int(max(0, round(x1s))); y1c = int(max(0, round(y1s)))
    x2c = int(min(img_w, round(x2s))); y2c = int(min(img_h, round(y2s)))
    if x2c <= x1c or y2c <= y1c:
        return None
    # decupez partea valida din imagine:
    crop = img_np[y1c:y2c, x1c:x2c].copy()
    pl_i, pt_i, pr_i, pb_i = (int(round(v)) for v in (pl, pt, pr, pb))
    # aici adaug acel padding cu pixeli negri:
    if pl_i > 0 or pt_i > 0 or pr_i > 0 or pb_i > 0:
        crop = np.pad(crop, ((pt_i, pb_i), (pl_i, pr_i), (0, 0)),
                      mode="constant", constant_values=0)
    pil_crop = Image.fromarray(crop).resize((size, size), Image.BICUBIC)
    tensor = effnet_transform(pil_crop).unsqueeze(0)
    return tensor

In [ ]:
def load_ground_truth(images_dir, labels_dir):
    gt = defaultdict(list)
    for lbl_path in sorted(labels_dir.glob("*.txt")):
        img_path = None
        for ext in IMG_EXTENSIONS:
            c = images_dir / (lbl_path.stem + ext)
            if c.exists(): img_path = c; break
        if not img_path:
            continue
        with Image.open(img_path) as pil:
            w, h = pil.size
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cid = int(parts[0])
                cx_n, cy_n, bw_n, bh_n = map(float, parts[1:5])
                x1 = (cx_n - bw_n / 2) * w; y1 = (cy_n - bh_n / 2) * h
                x2 = (cx_n + bw_n / 2) * w; y2 = (cy_n + bh_n / 2) * h
                gt[img_path.name].append([cid, x1, y1, x2, y2])
    total = sum(len(v) for v in gt.values())
    print(f"{len(gt)} imagini | {total} bbox-uri")
    return dict(gt)

In [ ]:
def iou_calc(a, b):
    xa = max(a[0], b[0]); ya = max(a[1], b[1])
    xb = min(a[2], b[2]); yb = min(a[3], b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    if inter == 0: return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)


def compute_ap(rec, prec):
    m_rec = np.concatenate(([0.0], rec, [1.0]))
    m_pre = np.concatenate(([1.0], prec, [0.0]))
    for i in range(len(m_pre) - 2, -1, -1):
        m_pre[i] = max(m_pre[i], m_pre[i + 1])
    idx = np.where(m_rec[1:] != m_rec[:-1])[0] + 1
    return float(np.sum((m_rec[idx] - m_rec[idx - 1]) * m_pre[idx]))


In [ ]:
def eval_class(preds_per_img, gt_per_img, class_id, iou_thresh):
    all_dets = []
    for fname, boxes in preds_per_img.items():
        for b in boxes:
            if int(b[5]) == class_id:
                all_dets.append((float(b[4]), fname, b[:4]))

    all_gt = {}; gt_used = {}; num_gt = 0
    for fname, boxes in gt_per_img.items():
        cls_boxes = [b[1:5] for b in boxes if b[0] == class_id]
        all_gt[fname] = cls_boxes; gt_used[fname] = set()
        num_gt += len(cls_boxes)

    if num_gt == 0:
        return {"ap": 0.0, "prec": 0.0, "rec": 0.0, "f1": 0.0,
                "tp": 0, "fp": len(all_dets), "fn": 0}
    if not all_dets:
        return {"ap": 0.0, "prec": 0.0, "rec": 0.0, "f1": 0.0,
                "tp": 0, "fp": 0, "fn": num_gt}

    all_dets.sort(key=lambda x: x[0], reverse=True)
    tp_arr = np.zeros(len(all_dets)); fp_arr = np.zeros(len(all_dets))

    for i, (score, fname, pred_box) in enumerate(all_dets):
        gt_boxes = all_gt.get(fname, [])
        best_iou, best_idx = -1, -1
        for j, gt_box in enumerate(gt_boxes):
            if j in gt_used[fname]: continue
            v = iou_calc(pred_box, gt_box)
            if v > best_iou: best_iou, best_idx = v, j
        if best_iou >= iou_thresh and best_idx != -1:
            tp_arr[i] = 1; gt_used[fname].add(best_idx)
        else:
            fp_arr[i] = 1

    cum_tp = np.cumsum(tp_arr); cum_fp = np.cumsum(fp_arr)
    rec_c  = cum_tp / num_gt
    pre_c  = cum_tp / (cum_tp + cum_fp + 1e-9)
    ap     = compute_ap(rec_c, pre_c)
    tp     = int(cum_tp[-1]); fp = int(cum_fp[-1]); fn = num_gt - tp
    rec    = tp / (tp + fn + 1e-9); prec = tp / (tp + fp + 1e-9)
    f1     = 2 * prec * rec / (prec + rec + 1e-9)
    return {"ap": ap, "prec": prec, "rec": rec, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn}


In [ ]:
def compute_all_metrics(preds, gt):
    print("\n  Metrici per clasa @ IoU=0.50:")
    per_class = {}
    for cid, cname in enumerate(CLASS_NAMES):
        r = eval_class(preds, gt, cid, IOU_THRESH_50)
        per_class[cname] = r
        print(f"    {cname:12s}  AP@50={r['ap']:.3f}  "
              f"P={r['prec']:.3f}  R={r['rec']:.3f}  "
              f"F1={r['f1']:.3f}  TP={r['tp']}  FP={r['fp']}  FN={r['fn']}")

    aps50    = [v["ap"] for v in per_class.values()]
    mAP50    = float(np.mean(aps50))
    aps_range = []
    for iou_t in IOU_THRESHOLDS_95:
        aps_at_t = [eval_class(preds, gt, cid, iou_t)["ap"]
                    for cid in range(NUM_CLASSES)]
        aps_range.append(np.mean(aps_at_t))
    mAP50_95  = float(np.mean(aps_range))
    prec_mean = float(np.mean([v["prec"] for v in per_class.values()]))
    rec_mean  = float(np.mean([v["rec"]  for v in per_class.values()]))
    f1_mean   = float(np.mean([v["f1"]   for v in per_class.values()]))
    total_tp  = sum(v["tp"] for v in per_class.values())
    total_fp  = sum(v["fp"] for v in per_class.values())
    total_fn  = sum(v["fn"] for v in per_class.values())
    return {
        "mAP50": mAP50, "mAP50_95": mAP50_95,
        "precision": prec_mean, "recall": rec_mean, "f1": f1_mean,
        "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn,
        "per_class": per_class,
    }


In [ ]:
import torch
def load_efficientnet(model_path: str, device: torch.device):
    checkpoint  = torch.load(model_path, map_location=device)
    arch        = checkpoint.get("arch", "efficientnet_b0")
    num_classes = len(checkpoint.get("classes", CLASS_NAMES))
    model = timm.create_model(arch, pretrained=False, num_classes=num_classes)
    model.load_state_dict(checkpoint["model_state"])
    model = model.to(device).eval()

    return model


In [ ]:
from torchvision import transforms

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

effnet_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [ ]:
import argparse, json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict

import cv2
import torch
import torch.nn.functional as F
from torchvision import transforms
import timm
device = 'cuda' if torch.cuda.is_available() else 'cpu'

yolo_single = YOLO('/content/drive/MyDrive/YOLO without P5 head/runs/yolov8m_singleclass_localizator2/weights/best.pt')
effnet      = load_efficientnet('/content/drive/MyDrive/YOLO without P5 head/runs/yolov8m_singleclass_localizator2/EfficientNet/best.pth', device)

CONFS = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
IOUS  = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]

def pipeline_predict(images_dir, conf, iou, augment):
    preds = {}
    for r in yolo_single.predict(source=images_dir, imgsz=640,
                                 conf=conf, iou=iou, augment = augment, stream=True, verbose=False):
        fname = Path(r.path).name
        b = r.boxes
        if b is None or len(b) == 0:
            preds[fname] = []; continue
        xyxy = b.xyxy.cpu().numpy(); ys = b.conf.cpu().numpy()
        img = np.array(Image.open(r.path).convert('RGB'))
        crops, keep = [], []
        for i, (x1, y1, x2, y2) in enumerate(xyxy):
            t = make_square_crop(img, x1, y1, x2, y2)   # tensor [1,3,224,224] sau None
            if t is None: continue
            crops.append(t); keep.append(i)
        if not crops:
            preds[fname] = []; continue
        with torch.no_grad():
            p = torch.softmax(effnet(torch.cat(crops).to(device)), 1).cpu().numpy()
        cls, cc = p.argmax(1), p.max(1)
        preds[fname] = [[*xyxy[keep[j]], float(ys[keep[j]] * cc[j]), int(cls[j])]
                        for j in range(len(keep))]
    return preds



# **Grid Search Pipeline**

In [ ]:
import argparse, json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict

import cv2
import torch
import torch.nn.functional as F
from torchvision import transforms
import timm
device = 'cuda' if torch.cuda.is_available() else 'cpu'

yolo_single = YOLO('/content/drive/MyDrive/YOLO without P5 head/runs/yolov8m_singleclass_localizator2/weights/best.pt')
effnet      = load_efficientnet('/content/drive/MyDrive/YOLO without P5 head/runs/yolov8m_singleclass_localizator2/EfficientNet/best.pth', device)

CONFS = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
IOUS  = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]

def pipeline_predict(images_dir, conf, iou, augment = false):
    preds = {}
    for r in yolo_single.predict(source=images_dir, imgsz=640,
                                 conf=conf, iou=iou, augment = true, stream=True, verbose=False):
        fname = Path(r.path).name              # cheie cu extensie (ca load_ground_truth)
        b = r.boxes
        if b is None or len(b) == 0:
            preds[fname] = []; continue
        xyxy = b.xyxy.cpu().numpy(); ys = b.conf.cpu().numpy()
        img = np.array(Image.open(r.path).convert('RGB'))
        crops, keep = [], []
        for i, (x1, y1, x2, y2) in enumerate(xyxy):
            t = make_square_crop(img, x1, y1, x2, y2)   # tensor [1,3,224,224] sau None
            if t is None: continue
            crops.append(t); keep.append(i)
        if not crops:
            preds[fname] = []; continue
        with torch.no_grad():
            p = torch.softmax(effnet(torch.cat(crops).to(device)), 1).cpu().numpy()
        cls, cc = p.argmax(1), p.max(1)
        preds[fname] = [[*xyxy[keep[j]], float(ys[keep[j]] * cc[j]), int(cls[j])]
                        for j in range(len(keep))]
    return preds

gt_val = load_ground_truth(Path(VAL_IMAGES), Path(VAL_LABELS))

res = []
for conf in CONFS:
    for iou in IOUS:
        print(f"Conf: {conf}, IoU: {iou}")
        m = compute_all_metrics(pipeline_predict(VAL_IMAGES, conf, iou), gt_val)
        res.append({'conf': conf, 'iou': iou, 'F1': m['f1'], 'P': m['precision'],
                    'R': m['recall'], 'mAP50': m['mAP50'], 'mAP50_95': m['mAP50_95']})
        print(f"  conf={conf:.2f} iou={iou:.2f} | F1={m['f1']:.4f} "
              f"P={m['precision']:.4f} R={m['recall']:.4f} mAP50={m['mAP50']:.4f}")

dfp = pd.DataFrame(res).sort_values('F1', ascending=False).reset_index(drop=True)
print("\nTOP 10 dupa F1")
print(dfp.head(10).to_string(index=False))

best = dfp.iloc[0]
PIPE_CONF, PIPE_IOU = float(best['conf']), float(best['iou'])
print(f"\nPipeline optim (F1): conf={PIPE_CONF}, iou={PIPE_IOU}, F1={best['F1']:.4f}")

357 imagini | 4495 bbox-uri
Conf: 0.01, IoU: 0.1

  Metrici per clasa @ IoU=0.50:
    comedo        AP@50=0.257  P=0.087  R=0.479  F1=0.147  TP=584  FP=6123  FN=634
    nodul         AP@50=0.082  P=0.141  R=0.208  F1=0.168  TP=200  FP=1215  FN=761
    papule        AP@50=0.206  P=0.120  R=0.500  F1=0.193  TP=898  FP=6610  FN=898
    pustule       AP@50=0.145  P=0.103  R=0.358  F1=0.160  TP=186  FP=1612  FN=334
  conf=0.01 iou=0.10 | F1=0.1673 P=0.1129 R=0.3863 mAP50=0.1725
Conf: 0.01, IoU: 0.2

  Metrici per clasa @ IoU=0.50:
    comedo        AP@50=0.261  P=0.087  R=0.504  F1=0.148  TP=614  FP=6442  FN=604
    nodul         AP@50=0.086  P=0.128  R=0.227  F1=0.163  TP=218  FP=1489  FN=743
    papule        AP@50=0.208  P=0.114  R=0.517  F1=0.187  TP=929  FP=7207  FN=867
    pustule       AP@50=0.148  P=0.100  R=0.369  F1=0.158  TP=192  FP=1719  FN=328
  conf=0.01 iou=0.20 | F1=0.1642 P=0.1073 R=0.4044 mAP50=0.1757
Conf: 0.01, IoU: 0.3

  Metrici per clasa @ IoU=0.50:
    comedo        

In [ ]:
def pipeline_predict(images_dir, conf, iou, augment=False):
    preds = {}
    for r in yolo_single.predict(source=images_dir, imgsz=640, conf=conf, iou=iou, augment=augment, stream=True, verbose=False):

        fname = Path(r.path).name
        b = r.boxes
        if b is None or len(b) == 0:
            preds[fname] = []; continue
        xyxy = b.xyxy.cpu().numpy(); ys = b.conf.cpu().numpy()
        img = np.array(Image.open(r.path).convert('RGB'))
        crops, keep = [], []
        for i, (x1, y1, x2, y2) in enumerate(xyxy):
            t = make_square_crop(img, x1, y1, x2, y2)
            if t is None: continue
            crops.append(t); keep.append(i)
        if not crops:
            preds[fname] = []; continue
        with torch.no_grad():
            p = torch.softmax(effnet(torch.cat(crops).to(device)), 1).cpu().numpy()
        cls, cc = p.argmax(1), p.max(1)
        preds[fname] = [[*xyxy[keep[j]], float(ys[keep[j]] * cc[j]), int(cls[j])] for j in range(len(keep))]
    return preds

# **Evaluare Pipeline cu conf = 0.15, iou = 0.45 pe TEST set**

In [ ]:
TEST_IMAGES = '/content/dataset/test/images'
TEST_LABELS = '/content/dataset/test/labels'

gt_test = load_ground_truth(Path(TEST_IMAGES), Path(TEST_LABELS))
PIPE_CONF = 0.15
PIPE_IOU = 0.45
print(f"EVALUARE FINALA PIPELINE PE TEST  (conf={PIPE_CONF}, iou={PIPE_IOU})\n")

for tta in [False, True]:
    tag = "TTA OFF" if tta == False else "TTA ON"
    preds = pipeline_predict(TEST_IMAGES, PIPE_CONF, PIPE_IOU, tta)
    m = compute_all_metrics(preds, gt_test)     # printeaza si per-clasa @IoU 0.50
    print(f"\n[{tag}]  F1={m['f1']:.4f}  P={m['precision']:.4f}  R={m['recall']:.4f}  "
          f"mAP50={m['mAP50']:.4f}  mAP50-95={m['mAP50_95']:.4f}")
    print(f"          TP/FP/FN = {m['total_tp']} / {m['total_fp']} / {m['total_fn']}\n")

358 imagini | 3422 bbox-uri
EVALUARE FINALA PIPELINE PE TEST  (conf=0.15, iou=0.45)


  Metrici per clasa @ IoU=0.50:
    comedo        AP@50=0.218  P=0.337  R=0.394  F1=0.363  TP=363  FP=715  FN=559
    nodul         AP@50=0.133  P=0.318  R=0.223  F1=0.262  TP=160  FP=343  FN=557
    papule        AP@50=0.212  P=0.321  R=0.404  F1=0.358  TP=552  FP=1167  FN=814
    pustule       AP@50=0.170  P=0.277  R=0.329  F1=0.301  TP=137  FP=357  FN=280

[TTA OFF]  F1=0.3210  P=0.3133  R=0.3374  mAP50=0.1834  mAP50-95=0.0643
          TP/FP/FN = 1212 / 2582 / 2210


  Metrici per clasa @ IoU=0.50:
    comedo        AP@50=0.242  P=0.304  R=0.443  F1=0.361  TP=408  FP=932  FN=514
    nodul         AP@50=0.179  P=0.299  R=0.297  F1=0.298  TP=213  FP=499  FN=504
    papule        AP@50=0.232  P=0.275  R=0.461  F1=0.345  TP=630  FP=1658  FN=736
    pustule       AP@50=0.185  P=0.258  R=0.365  F1=0.302  TP=152  FP=438  FN=265

[TTA ON]  F1=0.3264  P=0.2842  R=0.3913  mAP50=0.2095  mAP50-95=0.0724
     

# **Evaluare YOLO singleclass**

In [ ]:
m = yolo_single.val(
    data='/content/data.yaml',
    split='test',
    imgsz=640,
    single_cls=True,
    verbose=False,
    plots=False,
)

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 810.1±307.6 MB/s, size: 24.6 KB)
val: Scanning /content/dataset/test/labels... 358 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 358/358 1.6Kit/s 0.2s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 1.7it/s 13.3s
                   all        358       3422      0.453      0.435      0.404      0.147
Speed: 1.1ms preprocess, 25.8ms inference, 0.0ms loss, 1.6ms postprocess per image


# **Evaluare si YOLO si Pipeline cu aceleasi funcii - Tot yolo este superior**

In [ ]:
import io, contextlib
import numpy as np, pandas as pd
from pathlib import Path

VAL_IMAGES = '/content/dataset/val/images'
VAL_LABELS = '/content/dataset/val/labels'

def silent_metrics(preds, gt):
    with contextlib.redirect_stdout(io.StringIO()):     # compute_all_metrics printeaza mult
        return compute_all_metrics(preds, gt)

gt_val = load_ground_truth(Path(VAL_IMAGES), Path(VAL_LABELS))

rows = []
for conf in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    for iou in [0.30, 0.40, 0.45, 0.50, 0.60]:
        print(f"Conf:{conf} IoU:{iou}")
        m = silent_metrics(multiclass_predict(VAL_IMAGES, conf, iou), gt_val)
        rows.append({'conf':conf, 'iou':iou, 'F1':m['f1'],
                     'P':m['precision'], 'R':m['recall'], 'mAP50':m['mAP50']})

dfm = pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)
print(dfm.head(10).to_string(index=False))
OP_CONF, OP_IOU = float(dfm.iloc[0]['conf']), float(dfm.iloc[0]['iou'])
print(f"\nPunct de operare REAL multiclass: conf={OP_CONF}, iou={OP_IOU}")

357 imagini | 4495 bbox-uri
Conf:0.05 IoU:0.3
Conf:0.05 IoU:0.4
Conf:0.05 IoU:0.45
Conf:0.05 IoU:0.5
Conf:0.05 IoU:0.6
Conf:0.1 IoU:0.3
Conf:0.1 IoU:0.4
Conf:0.1 IoU:0.45
Conf:0.1 IoU:0.5
Conf:0.1 IoU:0.6
Conf:0.15 IoU:0.3
Conf:0.15 IoU:0.4
Conf:0.15 IoU:0.45
Conf:0.15 IoU:0.5
Conf:0.15 IoU:0.6
Conf:0.2 IoU:0.3
Conf:0.2 IoU:0.4
Conf:0.2 IoU:0.45
Conf:0.2 IoU:0.5
Conf:0.2 IoU:0.6
Conf:0.25 IoU:0.3
Conf:0.25 IoU:0.4
Conf:0.25 IoU:0.45
Conf:0.25 IoU:0.5
Conf:0.25 IoU:0.6
Conf:0.3 IoU:0.3
Conf:0.3 IoU:0.4
Conf:0.3 IoU:0.45
Conf:0.3 IoU:0.5
Conf:0.3 IoU:0.6
Conf:0.35 IoU:0.3
Conf:0.35 IoU:0.4
Conf:0.35 IoU:0.45
Conf:0.35 IoU:0.5
Conf:0.35 IoU:0.6
Conf:0.4 IoU:0.3
Conf:0.4 IoU:0.4
Conf:0.4 IoU:0.45
Conf:0.4 IoU:0.5
Conf:0.4 IoU:0.6
 conf  iou       F1        P        R    mAP50
 0.10 0.50 0.333371 0.308793 0.374336 0.200890
 0.10 0.30 0.332659 0.316560 0.364833 0.198219
 0.10 0.40 0.332213 0.313487 0.366980 0.198809
 0.10 0.45 0.331749 0.310455 0.369182 0.199365
 0.15 0.30 0.330255 0.371281 

In [ ]:
import pandas as pd
from pathlib import Path

def multiclass_predict(images_dir, conf, iou, augment=False):
    preds = {}
    for r in model_multi.predict(source=images_dir, imgsz=640, conf=conf, iou=iou,
                                 augment=augment, stream=True, verbose=False):
        fname = Path(r.path).name
        b = r.boxes
        if b is None or len(b) == 0:
            preds[fname] = []; continue
        xy = b.xyxy.cpu().numpy()
        sc = b.conf.cpu().numpy()
        cl = b.cls.cpu().numpy().astype(int)
        preds[fname] = [[*xy[i], float(sc[i]), int(cl[i])] for i in range(len(xy))]
    return preds

gt_test  = load_ground_truth(Path(TEST_IMAGES), Path(TEST_LABELS))
CONF_MAP = 0.01

BEST_IOU = 0.5
BEST_CONF = 0.1
PIPE_IOU = 0.45
PIPE_CONF = 0.15
mc_map = compute_all_metrics(multiclass_predict(TEST_IMAGES, CONF_MAP, BEST_IOU), gt_test)
pp_map = compute_all_metrics(pipeline_predict  (TEST_IMAGES, CONF_MAP, PIPE_IOU), gt_test)

mc_op  = compute_all_metrics(multiclass_predict(TEST_IMAGES, BEST_CONF, BEST_IOU), gt_test)
pp_op  = compute_all_metrics(pipeline_predict  (TEST_IMAGES, PIPE_CONF, PIPE_IOU), gt_test)

tab = pd.DataFrame([
    {'model':'YOLO multiclass', 'mAP50':mc_map['mAP50'], 'mAP50-95':mc_map['mAP50_95'],
     'F1@op':mc_op['f1'], 'P@op':mc_op['precision'], 'R@op':mc_op['recall']},
    {'model':'Pipeline hibrid', 'mAP50':pp_map['mAP50'], 'mAP50-95':pp_map['mAP50_95'],
     'F1@op':pp_op['f1'], 'P@op':pp_op['precision'], 'R@op':pp_op['recall']},
])
print(tab.to_string(index=False))

358 imagini | 3422 bbox-uri

  Metrici per clasa @ IoU=0.50:
    comedo        AP@50=0.338  P=0.101  R=0.708  F1=0.177  TP=653  FP=5815  FN=269
    nodul         AP@50=0.237  P=0.132  R=0.616  F1=0.217  TP=442  FP=2913  FN=275
    papule        AP@50=0.304  P=0.144  R=0.681  F1=0.237  TP=930  FP=5537  FN=436
    pustule       AP@50=0.224  P=0.179  R=0.400  F1=0.248  TP=167  FP=764  FN=250

  Metrici per clasa @ IoU=0.50:
    comedo        AP@50=0.257  P=0.085  R=0.572  F1=0.148  TP=527  FP=5683  FN=395
    nodul         AP@50=0.175  P=0.118  R=0.410  F1=0.183  TP=294  FP=2208  FN=423
    papule        AP@50=0.268  P=0.100  R=0.661  F1=0.174  TP=903  FP=8119  FN=463
    pustule       AP@50=0.187  P=0.110  R=0.420  F1=0.175  TP=175  FP=1413  FN=242

  Metrici per clasa @ IoU=0.50:
    comedo        AP@50=0.303  P=0.296  R=0.534  F1=0.381  TP=492  FP=1170  FN=430
    nodul         AP@50=0.180  P=0.309  R=0.364  F1=0.334  TP=261  FP=583  FN=456
    papule        AP@50=0.250  P=0.343  R=0.4